# VIDI 75-Video Research Pipeline
### ISRO Internship Project — VLM Disaster Analyzer

**Objective:** Download exactly 75 disaster videos from the VIDI dataset
(5 categories × 15 videos) and prepare them for VLM evaluation.

| Category | VIDI Labels Used | Videos |
|---|---|---|
| Flood | flooded, heavy_rainfall, storm_surge | 15 |
| Wildfire | wildfire, on_fire, fire_whirl | 15 |
| Earthquake | earthquake, collapsed | 15 |
| Landslide | landslide, mudslide_mudflow, rockslide_rockfall | 15 |
| Cyclone | tropical_cyclone, tornado, derecho | 15 |
| **Total** | | **75** |

**Workflow:**
```
VIDI GitHub CSVs → Select 75 clips → Download + Trim → Extract Frames → Manifests → VLM Evaluation
```

**Storage:** All data is saved to Google Drive at `MyDrive/ISRO_Project/` and persists across Colab sessions.

---
## Cell 1 — Install System Dependencies

In [ ]:
# Install yt-dlp (YouTube downloader) and Python packages
# ffmpeg is pre-installed on Colab but we confirm it here
!pip install -q yt-dlp openpyxl requests tqdm pillow
!apt-get install -y -q ffmpeg 2>/dev/null | tail -1

# Verify tools are available
import subprocess
for tool, cmd in [("yt-dlp", ["yt-dlp", "--version"]),
                   ("ffmpeg",  ["ffmpeg",  "-version"]),
                   ("ffprobe", ["ffprobe", "-version"])]:
    r = subprocess.run(cmd, capture_output=True, text=True)
    status = "✓" if r.returncode == 0 else "✗ MISSING"
    ver = r.stdout.splitlines()[0] if r.returncode == 0 else r.stderr[:60]
    print(f"  {status}  {tool}: {ver}")

print("\nDependencies ready.")

---
## Cell 2 — Mount Google Drive + Set Paths

In [ ]:
import os
from pathlib import Path

# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── Project paths ─────────────────────────────────────────────────────────────
# All research data goes here — persists across Colab sessions.
DRIVE_PROJECT = "/content/drive/MyDrive/ISRO_Project"

# The VLM repo is cloned to Colab local storage (Cell 3).
REPO_PATH = "/content/vlm-disaster-analyzer"

# Set environment variable so pipeline config reads the Drive path.
os.environ["VLM_PROJECT_ROOT"] = DRIVE_PROJECT

# ── Display layout ────────────────────────────────────────────────────────────
dataset_root = Path(DRIVE_PROJECT) / "datasets" / "video_dataset"
print("Project layout:")
for sub in [
    "datasets/video_dataset/raw_videos/{flood,wildfire,earthquake,landslide,cyclone}/",
    "datasets/video_dataset/extracted_frames/{category}/{video_stem}/frame_*.jpg",
    "datasets/video_dataset/metadata/video_manifest.csv",
    "datasets/video_dataset/metadata/frame_manifest.csv",
    "datasets/video_dataset/metadata/dataset_statistics.xlsx",
    "datasets/video_dataset/evaluation/Video_VLM_Comparison.xlsx",
]:
    print(f"  {DRIVE_PROJECT}/{sub}")

print(f"\nDrive mounted ✓")
print(f"PROJECT_ROOT  = {DRIVE_PROJECT}")

---
## Cell 3 — Clone Repository + Add to Python Path

In [ ]:
import sys
import os
from pathlib import Path

REPO_PATH = "/content/vlm-disaster-analyzer"

if not Path(REPO_PATH).exists():
    # Replace with your actual GitHub repository URL
    !git clone --depth 1 https://github.com/ujjesha1312/vlm-disaster-analyzer.git {REPO_PATH}
else:
    print(f"Repo already cloned: {REPO_PATH}")
    !git -C {REPO_PATH} pull --ff-only 2>/dev/null | tail -1

# Add pipeline scripts to Python path
PIPELINE_DIR = f"{REPO_PATH}/scripts/video_pipeline"
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)

# Verify imports work
import config as cfg
print(f"\nConfig loaded:")
print(f"  PROJECT_ROOT      = {cfg.PROJECT_ROOT}")
print(f"  DATASET_ROOT      = {cfg.DATASET_ROOT}")
print(f"  TARGET_CATEGORIES = {cfg.TARGET_CATEGORIES}")
print(f"  TOTAL_VIDEOS      = {cfg.TOTAL_VIDEOS}")

---
## Cell 4 — Create Directory Structure

In [ ]:
from pathlib import Path
import config as cfg

dirs_to_create = [
    cfg.RAW_VIDEOS_ROOT,
    cfg.FRAMES_ROOT,
    cfg.METADATA_DIR,
    cfg.EVALUATION_DIR,
    cfg.YT_CACHE_DIR,
] + [
    cfg.RAW_VIDEOS_ROOT / cat for cat in cfg.TARGET_CATEGORIES
] + [
    cfg.FRAMES_ROOT / cat for cat in cfg.TARGET_CATEGORIES
]

print("Creating directory structure...")
for d in dirs_to_create:
    d.mkdir(parents=True, exist_ok=True)
    status = "exists" if d.exists() else "FAILED"
    print(f"  [{status:^7}]  {d.relative_to(cfg.PROJECT_ROOT)}")

print("\nDirectory structure ready ✓")

---
## Cell 5 — Fetch VIDI Annotation CSVs + Select 75 Clips

Downloads 14 small CSV files (~KB each) from the VIDI GitHub repository
and selects exactly 15 high-quality clips per category.

**Selection criteria:**
- Duration: 10–300 seconds
- Prefer English-language clips
- Diversity: at most 2 clips per YouTube video ID

In [ ]:
from download_videos import build_selection
import pandas as pd

# Fetch VIDI CSVs + select 75 clips
# This takes ~30 seconds (downloads 14 small CSV files from GitHub)
selection = build_selection()

# Display selection details
print("\n" + "="*60)
print("  CLIP SELECTION SUMMARY")
print("="*60)
summary = (
    selection
    .groupby("research_category")
    .agg(
        clips          = ("clip_id",     "count"),
        unique_yt_ids  = ("youtube_id",  "nunique"),
        avg_duration_s = ("duration_s",  "mean"),
        english_clips  = ("lang",        lambda x: (x == "EN").sum()),
    )
    .reindex(pd.Index(["flood","wildfire","earthquake","landslide","cyclone"]))
)
summary["avg_duration_s"] = summary["avg_duration_s"].round(1)
print(summary.to_string())
print("-"*60)
print(f"  Total clips selected : {len(selection)}")
print(f"  Unique YouTube IDs   : {selection['youtube_id'].nunique()}")
print(f"  Saved to             : {selection.__class__.__module__}")

### Cell 5b — Preview Selection (optional)
Inspect the selected clips before downloading.

In [ ]:
import config as cfg
import pandas as pd

sel = pd.read_csv(cfg.METADATA_DIR / "selected_clips.csv")

display_cols = ["research_category", "vidi_label", "youtube_id",
                "time_start", "time_end", "duration_s", "lang"]

for cat in cfg.TARGET_CATEGORIES:
    print(f"\n── {cat.upper()} ({'─'*40})")
    sub = sel[sel["research_category"] == cat][display_cols]
    print(sub.to_string(index=False))

---
## Cell 6 — Download 75 Videos

**What happens here:**
1. For each unique YouTube ID in the selection, download the full video via `yt-dlp` (at ≤720p)
2. Trim each annotated clip using `ffmpeg` → `raw_videos/{category}/`
3. Delete the full YouTube video to save disk space

**Estimated time:** 45–90 minutes depending on video lengths and connection speed.

Already-downloaded clips are skipped automatically — safe to re-run after interruption.

In [ ]:
import time
from download_videos import run_download

t0 = time.perf_counter()

report = run_download(
    selection = None,    # loads from selected_clips.csv automatically
    workers   = 4,       # parallel downloads (safe for Colab)
    dry_run   = False,   # set True to simulate without downloading
)

elapsed = time.perf_counter() - t0

print(f"\n{'='*60}")
print(f"  Download complete in {elapsed/60:.1f} minutes")
print(f"  Succeeded : {(report['status']=='success').sum()}")
print(f"  Failed    : {(report['status']!='success').sum()}")
print(f"{'='*60}")

if (report["status"] != "success").any():
    print("\nFailed clips:")
    failed = report[report["status"] != "success"][["research_category","youtube_id","status","message"]]
    print(failed.to_string(index=False))

---
## Cell 7 — Verify Downloaded Videos

In [ ]:
import config as cfg
from pathlib import Path

VIDEO_EXT = {".mp4", ".avi", ".mov", ".mkv"}

print("Video counts per category:")
print("-" * 40)
total = 0
for cat in cfg.TARGET_CATEGORIES:
    folder = cfg.RAW_VIDEOS_ROOT / cat
    if not folder.exists():
        print(f"  {cat:<12}: folder missing")
        continue
    videos = [p for p in folder.iterdir() if p.suffix.lower() in VIDEO_EXT]
    sizes  = [p.stat().st_size / (1024**2) for p in videos]
    total_mb = sum(sizes)
    total   += len(videos)
    print(f"  {cat:<12}: {len(videos):>3} videos  |  {total_mb:>7.1f} MB")

print("-" * 40)
print(f"  {'TOTAL':<12}: {total:>3} videos")

if total < 75:
    missing = 75 - total
    print(f"\n  ⚠  {missing} video(s) missing — re-run Cell 6 to retry failed downloads.")
else:
    print("\n  ✓  All 75 videos present.")

---
## Cell 8 — Extract Frames

Samples 8 frames uniformly from each video and saves them as JPEGs.
75 videos × 8 frames = **600 frames** ready for VLM inference.

**Output:** `extracted_frames/{category}/{video_stem}/frame_001.jpg … frame_008.jpg`

In [ ]:
from extract_frames import run_extraction

frame_manifest = run_extraction(
    n         = 8,      # frames per video (set to 12 for more coverage)
    overwrite = False,  # set True to re-extract already-done videos
)

extracted = (frame_manifest["status"] == "extracted").sum()
cached    = (frame_manifest["status"] == "cached").sum()
failed    = (frame_manifest["status"] == "failed").sum()

print(f"\nFrame extraction complete:")
print(f"  Extracted : {extracted}")
print(f"  Cached    : {cached}")
print(f"  Failed    : {failed}")

### Cell 8b — Preview Extracted Frames (optional)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import config as cfg
from pathlib import Path
import random

# Show 8 frames from one randomly chosen video per category
fig, axes = plt.subplots(5, 8, figsize=(22, 14))
fig.suptitle("Extracted Frames — One Video per Category", fontsize=14, fontweight="bold")

for row_i, cat in enumerate(cfg.TARGET_CATEGORIES):
    frame_cat_dir = cfg.FRAMES_ROOT / cat
    if not frame_cat_dir.exists():
        continue
    video_dirs = sorted(frame_cat_dir.iterdir())
    if not video_dirs:
        continue
    # Pick a random video folder
    vid_dir  = random.choice(video_dirs)
    frames_f = sorted(vid_dir.glob("*.jpg"))

    for col_i in range(8):
        ax = axes[row_i][col_i]
        if col_i < len(frames_f):
            img = cv2.cvtColor(cv2.imread(str(frames_f[col_i])), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(f"{col_i+1}", fontsize=7)
        ax.axis("off")

    axes[row_i][0].set_ylabel(cat.capitalize(), fontsize=10, rotation=90, labelpad=8)

plt.tight_layout()
plt.savefig(str(cfg.METADATA_DIR / "frame_preview.png"), dpi=100, bbox_inches="tight")
plt.show()
print(f"Saved → {cfg.METADATA_DIR / 'frame_preview.png'}")

---
## Cell 9 — Generate Manifests + Statistics Excel

In [ ]:
from generate_excel import run_generate

run_generate(skip_frames=False)

import config as cfg
print("\nGenerated files:")
for f in [
    cfg.METADATA_DIR / "video_manifest.csv",
    cfg.METADATA_DIR / "frame_manifest.csv",
    cfg.METADATA_DIR / "dataset_statistics.xlsx",
]:
    exists = "✓" if f.exists() else "✗"
    size   = f"{f.stat().st_size/1024:.1f} KB" if f.exists() else "missing"
    print(f"  {exists}  {f.name:<35} ({size})")

---
## Cell 10 — Display Dataset Statistics

In [ ]:
import config as cfg
import pandas as pd
from datetime import timedelta

vm = pd.read_csv(cfg.METADATA_DIR / "video_manifest.csv")
fm = pd.read_csv(cfg.METADATA_DIR / "frame_manifest.csv")

print("=" * 62)
print("  VIDI RESEARCH DATASET — 75-VIDEO SUBSET STATISTICS")
print("=" * 62)
print(f"  Source          : VIDI Dataset (github.com/vididataset/VIDI)")
print(f"  Total videos    : {len(vm)}")
print(f"  Total frames    : {len(fm)}  ({len(fm)//max(len(vm),1)} per video)")
print(f"  Categories      : {', '.join(cfg.TARGET_CATEGORIES)}")

if "duration_seconds" in vm.columns and not vm["duration_seconds"].isna().all():
    healthy = vm.dropna(subset=["duration_seconds"])
    total_s = healthy["duration_seconds"].sum()
    print(f"  Total duration  : {str(timedelta(seconds=int(total_s)))}")
    print(f"  Avg duration    : {healthy['duration_seconds'].mean():.1f}s per video")

if "file_size_mb" in vm.columns:
    total_mb = vm["file_size_mb"].sum()
    print(f"  Total size      : {total_mb:.1f} MB  ({total_mb/1024:.2f} GB)")

print("\n── Per-category breakdown ──────────────────────────────────")
summary = vm.groupby("category").agg(
    videos          = ("video_id",         "count"),
    avg_duration_s  = ("duration_seconds",  "mean"),
    total_size_mb   = ("file_size_mb",      "sum"),
).reindex(cfg.TARGET_CATEGORIES).fillna(0)
summary["avg_duration_s"] = summary["avg_duration_s"].round(1)
summary["total_size_mb"]  = summary["total_size_mb"].round(1)
print(summary.to_string())
print("=" * 62)

---
## Cell 11 — VLM Evaluation (requires backend)

**Prerequisites:**
1. The VLM Disaster Analyzer backend must be running in **a separate Colab notebook**:
   ```python
   # In backend notebook:
   !python backend/main.py
   ```
2. Copy the ngrok URL from that notebook and paste it below.
3. For BLIP-2 and LLaVA, set: `ACTIVE_MODELS=clip,blip2,llava,qwen`

**Skip this cell** if you only want the video dataset (without VLM predictions).

In [ ]:
import os
import requests

# ── Set your backend URL here ─────────────────────────────────────────────────
BACKEND_URL = "https://YOUR-NGROK-URL.ngrok-free.app"  # ← paste your ngrok URL
# BACKEND_URL = "http://localhost:8000"               # if running on same machine

os.environ["VLM_BACKEND"] = BACKEND_URL

# Test connectivity
try:
    r = requests.get(f"{BACKEND_URL}/models", timeout=10)
    r.raise_for_status()
    models = r.json()
    print(f"  ✓ Backend reachable: {BACKEND_URL}")
    print(f"  Available models: {models.get('available_models', [])}")
except Exception as e:
    print(f"  ✗ Cannot reach backend: {e}")
    print("  → Start the backend and update BACKEND_URL above.")

In [ ]:
import time
from evaluate_videos import run_evaluation

# Choose models to evaluate:
#   ["clip"]              — fastest, zero GPU (production default)
#   ["clip", "qwen"]      — standard production pair
#   ["clip", "blip2", "llava", "qwen"]  — full research comparison
MODELS_TO_EVALUATE = ["clip", "qwen"]

t0 = time.perf_counter()

run_evaluation(
    models      = MODELS_TO_EVALUATE,
    backend_url = BACKEND_URL,
)

print(f"\nEvaluation complete in {(time.perf_counter()-t0)/60:.1f} minutes.")

---
## Cell 12 — Display Evaluation Results

In [ ]:
import config as cfg
import pandas as pd
from pathlib import Path

print("=" * 60)
print("  VLM EVALUATION RESULTS")
print("=" * 60)

eval_dir = cfg.EVALUATION_DIR
models   = ["clip", "blip2", "llava", "qwen"]

for model in models:
    csv = eval_dir / f"{model}_results.csv"
    if not csv.exists():
        print(f"  [{model.upper():<5}] Not evaluated yet.")
        continue

    df       = pd.read_csv(csv)
    acc      = df["correct"].mean() * 100 if not df.empty else 0
    avg_conf = df["avg_confidence"].mean() if "avg_confidence" in df.columns else None

    print(f"\n── {model.upper()} {'─'*40}")
    print(f"  Accuracy : {acc:.1f}%  ({df['correct'].sum()}/{len(df)})")
    if avg_conf is not None and pd.notna(avg_conf):
        print(f"  Avg Conf : {avg_conf:.1f}%")

    # Per-category breakdown
    cat_acc = (
        df.groupby("category")["correct"]
        .agg(correct="sum", total="count")
        .assign(accuracy=lambda x: (x["correct"]/x["total"]*100).round(1))
    )
    print(cat_acc.to_string())

# List generated files
print("\n── Generated files ─────────────────────────────────────")
for f in sorted(eval_dir.iterdir()):
    print(f"  {f.name:<45} ({f.stat().st_size/1024:.1f} KB)")

---
## Cell 13 — Final File Listing

In [ ]:
import config as cfg
from pathlib import Path

VIDEO_EXT = {".mp4", ".avi", ".mov", ".mkv"}

def _count(folder: Path, ext=None):
    if not folder.exists(): return 0
    files = list(folder.rglob("*"))
    return sum(1 for f in files if f.is_file() and (ext is None or f.suffix.lower() in ext))

print("=" * 60)
print("  FINAL DATASET SUMMARY")
print("=" * 60)

raw_total = sum(_count(cfg.RAW_VIDEOS_ROOT / c, VIDEO_EXT) for c in cfg.TARGET_CATEGORIES)
frm_total = sum(_count(cfg.FRAMES_ROOT / c, {".jpg",".png"}) for c in cfg.TARGET_CATEGORIES)

print(f"  Raw videos      : {raw_total}  (target: 75)")
print(f"  Extracted frames: {frm_total}  (target: ~600)")

print("\n── Google Drive layout ──────────────────────────────────")
for cat in cfg.TARGET_CATEGORIES:
    n_vid = _count(cfg.RAW_VIDEOS_ROOT / cat, VIDEO_EXT)
    n_frm = _count(cfg.FRAMES_ROOT / cat, {".jpg",".png"})
    print(f"  {cat:<12}  {n_vid:>3} videos  |  {n_frm:>4} frames")

print("\n── Metadata files ───────────────────────────────────────")
for f in sorted(cfg.METADATA_DIR.iterdir()):
    if f.is_file() and not f.name.startswith("."):
        print(f"  {f.name:<40} ({f.stat().st_size/1024:.1f} KB)")

print("\n── Evaluation files ─────────────────────────────────────")
if cfg.EVALUATION_DIR.exists():
    for f in sorted(cfg.EVALUATION_DIR.iterdir()):
        if f.is_file():
            print(f"  {f.name:<40} ({f.stat().st_size/1024:.1f} KB)")
else:
    print("  (no evaluation files yet — run Cell 11)")

print("\n" + "=" * 60)
print("  PIPELINE COMPLETE ✓")
print("=" * 60)